## Лаборатоная работа №2
#### Ковалев Андрей ИУ5-63Б
### Загрузка библиотек

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Загрузка и первичный анализ данных

In [7]:
data = pd.read_csv('rotten_tomatoes_critic_reviews.csv', sep=",")
print('Кол-во строк: ', data.shape[0],'\n' ,'Кол-во столбцов: ', data.shape[1], sep = '')
print(data.dtypes)
print('Пропуски по столбцам:', data.isnull().sum())
print(data.head())

Кол-во строк: 1130017
Кол-во столбцов: 8
rotten_tomatoes_link    object
critic_name             object
top_critic                bool
publisher_name          object
review_type             object
review_score            object
review_date             object
review_content          object
dtype: object
Пропуски по столбцам: rotten_tomatoes_link         0
critic_name              18529
top_critic                   0
publisher_name               0
review_type                  0
review_score            305936
review_date                  0
review_content           65806
dtype: int64
  rotten_tomatoes_link      critic_name  top_critic           publisher_name  \
0            m/0814255  Andrew L. Urban       False           Urban Cinefile   
1            m/0814255    Louise Keller       False           Urban Cinefile   
2            m/0814255              NaN       False      FILMINK (Australia)   
3            m/0814255     Ben McEachen       False  Sunday Mail (Australia)   
4            m

### Обработка пропусков в данных

In [97]:
# Исключим из датасета строки с одновременно пустыми полями 'review_score' b 'review_content'.
data_1 = data.copy()
data_1.dropna(axis=0, how='all', subset = ['review_score', 'review_content'])
print('Было: ', data.shape[0],'\n','Стало: ',data_1.shape[0], sep = '')
# Заменим пропуски в колонке critic_name на 'Noname' и в колонке review_content на 'No content'
data_1.fillna({'critic_name':'Noname', 'review_content':'No content'}, inplace = True)
print(data_1[(data_1['critic_name'] == 'Noname') | (data_1['review_content'] == 'No content')])
# Заменим пропуски в review_score на среднее значение для соотвествующей категории (Rotten или Fresh)
## Импортируем инструменты импьютации
from sklearn.impute import SimpleImputer
from sklearn.impute import MissingIndicator

def score2float(score) -> float:
    if(isinstance(score, float)): return score
    if '/' in score:
        part2 = float(score[score.index('/')+1:])
        if part2 == 0: return None
        return float(score[:score.index('/')])/part2
    else:
        marks = {'A':1.0,'B':0.8,'C':0.6,'D':0.4,'F':0.2 }
        for key in marks:
            if key in score:
                if '-' in score: return marks[key]-0.1
                else: return marks[key]

data_1["review_score"] = data_1["review_score"].apply(score2float)
data_1["review_score"].astype(float)
print(data_1["review_score"].info())
imputer = SimpleImputer(strategy = 'mean')
indicator = MissingIndicator()
data_segments = [data_1[data_1['review_type'] == 'Rotten'],data_1[data_1['review_type'] == 'Fresh']]
for i in range(len(data_segments)):
    score_col = data_segments[i][['review_score']]
    mask_missing= indicator.fit_transform(score_col)
    score_col = imputer.fit_transform(score_col)
    filled_data = score_col[mask_missing]
    print(filled_data[filled_data.size-1])
    data_segments[i]['review_score'] = score_col
data_1 = pd.concat(data_segments)
print('Пропуски по столбцам:', data_1.isnull().sum())



"""def test_num_impute_col(dataset, column, strategy_param):
    temp_data = dataset[[column]] # выбрали колонку из датасета
    
    indicator = MissingIndicator() #создали объект индикатора
    mask_missing_values_only = indicator.fit_transform(temp_data) #создали маску для сейчас пустых значений
    
    imp_num = SimpleImputer(strategy=strategy_param) #создали объект импьютера по стратегии
    data_num_imp = imp_num.fit_transform(temp_data)  #заменили в колонке
    
    filled_data = data_num_imp[mask_missing_values_only] #по маске получили вставленные данные
    
    return column, strategy_param, filled_data.size, filled_data[0], filled_data[filled_data.size-1] # вернули, что набрали"""

Было: 1130017
Стало: 1130017
        rotten_tomatoes_link     critic_name  top_critic       publisher_name  \
2                  m/0814255          Noname       False  FILMINK (Australia)   
80                 m/0814255          Noname       False        National Post   
101                m/0814255          Noname        True             Time Out   
182                m/0878835          Noname       False        National Post   
283                m/0878835  Ben Kenigsberg        True             Time Out   
...                      ...             ...         ...                  ...   
1130006               m/zulu          Noname       False      Empire Magazine   
1130009          m/zulu_dawn    Emanuel Levy       False      EmanuelLevy.Com   
1130010          m/zulu_dawn  Brandon Judell       False             PopcornQ   
1130011          m/zulu_dawn    Cole Smithey       False      ColeSmithey.com   
1130012          m/zulu_dawn   Chuck O'Leary       False     Fantastica Daily   

C:\Users\Dru09\AppData\Local\Temp\ipykernel_7708\2415607186.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_segments[i]['review_score'] = score_col
C:\Users\Dru09\AppData\Local\Temp\ipykernel_7708\2415607186.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_segments[i]['review_score'] = score_col


Пропуски по столбцам: rotten_tomatoes_link    0
critic_name             0
top_critic              0
publisher_name          0
review_type             0
review_score            0
review_date             0
review_content          0
dtype: int64


'def test_num_impute_col(dataset, column, strategy_param):\n    temp_data = dataset[[column]] # выбрали колонку из датасета\n\n    indicator = MissingIndicator() #создали объект индикатора\n    mask_missing_values_only = indicator.fit_transform(temp_data) #создали маску для сейчас пустых значений\n\n    imp_num = SimpleImputer(strategy=strategy_param) #создали объект импьютера по стратегии\n    data_num_imp = imp_num.fit_transform(temp_data)  #заменили в колонке\n\n    filled_data = data_num_imp[mask_missing_values_only] #по маске получили вставленные данные\n\n    return column, strategy_param, filled_data.size, filled_data[0], filled_data[filled_data.size-1] # вернули, что набрали'